In [ ]:
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import os
import torchvision.transforms.functional as TF
import random
import shutil
import cv2
import numpy as np
from PIL import Image
import glob
import timm

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from timm.models.vision_transformer import Attention

#this is how you use your own data in google drive
from google.colab import drive


#define device - put this in the main func
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1 . Mount the Google Drive
drive.mount('/content/drive')



# 2. path for saving the weights
weights_path = "/content/drive/MyDrive/FinalProject_CSC2503/ViTmiX_pretrained_weights.pth"


### CREATING A ViTmiX Vision Transformer (pretrained) for Fracture Dataset ###



# create a class to inherit form timm's Attention omdule -
# change forward method SO WE SAVE ATTENTION MAP SO WE CAN USE IT LATER
class AttentionWithHook(Attention):
  def __init__(self, *args, **kwargs):
    super().__init__(*args, **kwargs)
    self.last_attention = None


  def forward(self, x, **kwargs): # x is input tokens [B=Batch size, N=number of tokens/patches, C=embedding dim]

    B, N, C = x.shape # unpack dimensions
    qkv = self.qkv(x) # linear layer that produces all queries, keys, values AT ONCE - output is [B, N, C*3]
    qkv = qkv.reshape(B, N, 3, self.num_heads, C // self.num_heads)
    q, k,v = qkv.permute(2, 0, 3, 1, 4) # q, k, v each have shape of [B, num_heads, N, head_dim] permute here rearranges to: 3=Q/K/V, 1=batch, 2=heads, 3=tokens, 4=head_dim

    # compute attention here with scaled-dot-product
    attn = (q @ k.transpose(-2, -1))*self.scale
    attn = attn.softmax(dim=-1)
    # STORE ATTENTION FOR VISUALIZATION PLOTS LATER
    self.last_attention = attn.detach()

    x = (attn @ v).transpose(1, 2).reshape(B, N, C)
    x = self.proj(x)
    x = self.proj_drop(x)
    return x


# Mixes tokens between img1 and img2, creates 12 transformer blocks
class ViT(nn.Module):
  def __init__(self, mix_ratio=0.1):
    super().__init__()

    self.model = timm.create_model('vit_tiny_patch16_224', pretrained=True)
    self.mix_ratio = mix_ratio

    # REPLACE all Attention omdulesin VIT tiny
    for blk in self.model.blocks:

      old_attn = blk.attn # original pretrained attention module

      new_attn = AttentionWithHook(
          dim = old_attn.qkv.weight.shape[1],
          num_heads = old_attn.num_heads,
          qkv_bias = (old_attn.qkv.bias is not None),
          attn_drop = old_attn.attn_drop.p,
          proj_drop = old_attn.proj_drop.p
      )

      # copy pretrained weights into new module
      new_attn.qkv.weight.data = old_attn.qkv.weight.data.clone()
      new_attn.qkv.bias.data = old_attn.qkv.bias.data.clone()
      new_attn.proj.weight.data = old_attn.proj.weight.data.clone()
      new_attn.proj.bias.data = old_attn.proj.bias.data.clone()


      # REPLACE ATTN IN THE BLOCK
      blk.attn = new_attn



  # define the vit mixing  function that mixes the patches (NOT IN A CLASS RN)
  def vitmix(self, tokens1, tokens2):

    #[B, N, D] (B= Batch, N=#patches, D = embedded dim)
    # mix_Ratio is a float between [0, 1], it's 0.5 now
    #returns the mixed tokens (token2 mixed into token1)

    B, N, D = tokens1.shape
    num_mix = int(N*self.mix_ratio) #num of patches to replace ex. 196*(0.5)=98
    idx = torch.randperm(N, device = tokens1.device)[:num_mix] # returns a random permutation of integers from 0 to N-1 and we take the first num_mix # of those integers (so not all integers, just num_mix of them). its the random patch indicies out of N-1

    # mixed starts as a complete copy of tokens1 and contains ONLY tokens1
    mixed = tokens1.clone() # clone to keep an extra copy of tokens1 so the next line doesnt modify the values in the tokens1 memory

    # idx contains patch indicies selected randomly
    # we r replacing the tokens from img1 at idx with the corresponding tokens from img2
    mixed[:, idx] = tokens2[:, idx] # at every selected patch position, idx copy the patch embedding from tokens2 into mixed
    return mixed


  def forward(self, img1, img2):

    # move img to device
    img1 = img1.to(device)
    img2 = img2.to(device)


    # preprocess through patch embedding
    x1 = self.model.patch_embed(img1) #[B, N, D]
    x2 = self.model.patch_embed(img2)

    # add the class token
    cls_token = self.model.cls_token.expand(x1.shape[0], -1, -1).clone()
    x1 = torch.cat((cls_token, x1), dim=1)
    x2 = torch.cat((cls_token, x2), dim=1)

    # add positional embedding
    x1 = x1 + self.model.pos_embed.to(img1.device)
    x2 = x2 + self.model.pos_embed.to(img2.device)

    # vitmix - skip index 0 bc the class token is there and WE DONT MIX THE CLASS TOKEN
    mixed_tokens = self.vitmix(x1[:, 1:], x2[:, 1:])
    mixed = torch.cat([x1[:, :1], mixed_tokens], dim=1)

    for blk in self.model.blocks:
      mixed = blk(mixed)

    # classification head
    mixed = self.model.norm(mixed)
    cls = mixed[:, 0]


    # return the final classification head result
    return self.model.head(cls)



# PREPROCESSING IMGS - SCALE BETWEEN 640X640, REFLECTIVE PIXEL FILLING TO CREATE 640X640 DIMENSIONS
def preprocess_img_resize_pad(img, target_dim = 224):

  # 1. extract the width and height
  w, h = img.size

  # 2. calculate the scale factor needed (NO DISTORTION USING THIS SCALE FACTOR)
  scale_factor = target_dim/max(h, w) # div 640 by max(h, w)

  # 3. rescale and convert result to int
  new_w = int(scale_factor*w)
  new_h = int(scale_factor*h)

  # 4. resize orig image
  img = img.resize((new_w, new_h), Image.BILINEAR)


  # 5. calculate how much reflective padding we'll need
  pad_w = target_dim - new_w
  pad_h = target_dim - new_h

  # 6. want image centered before padding - tht's why we do integer // div pad_w and pad_h by 2
  # if odd num like 241, we know 241//2 = 120 and 241-120 =121 so left side gets 120 block of pixels and right side gets 121 block of pixels (no iteration of the pixels)
  # padding = ( LEFT, TOP, RIGHT, BOTTOM)
  # (0, 0) is top left, (W, 0) top right, (0, H) bottom left, (W, H) bottom right
  padding = (pad_w // 2, pad_h//2, pad_w - pad_w//2, pad_h - pad_h//2)

  # 7. REFLECTIVE PADDING ADDED HERE - fill = 0 and padding_mode="constant" make padding black
  img = TF.pad(img, padding, fill=0, padding_mode="constant")


  return img




################ VISUALIZATION FUNCTIONS ###################

visualization_dir = "/content/drive/MyDrive/FinalProject_CSC2503/ViTmiX_pretrained_Visualizations_NEW"
visualization_dir_plots = "/content/drive/MyDrive/FinalProject_CSC2503/ViTmiX__pretrained_Visualizations_Plots_NEW"

def show_rollout_on_image(img, rollout, save_path= visualization_dir):
    """
    img: PIL Image
    rollout: numpy array (H', W')
    """
    img_np = np.array(img).astype(np.float32)/255.0
    if img_np.ndim == 2:
        img_np = np.stack([img_np]*3, axis=-1)  # convert grayscale to 3 channels

    # resize rollout to original image size
    heatmap = cv2.resize(rollout, (img_np.shape[1], img_np.shape[0]))

    # normalize heatmap
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    # apply colormap
    heatmap_color = cv2.applyColorMap(np.uint8(255*heatmap), cv2.COLORMAP_JET)
    heatmap_color = heatmap_color[..., ::-1] / 255.0  # BGR->RGB

    overlay = 0.5*img_np + 0.5*heatmap_color
    overlay_img = Image.fromarray((overlay*255).astype(np.uint8))

    return overlay_img


def attention_rollout(model, x, head_fusion='mean'):
    """
    Compute attention rollout for a ViT or ViTMix model.
    """
    model.eval()
    attn_weights = []

    # register hooks to capture attention
    hooks = []
    for blk in model.model.blocks: #transformer.blocks:  # adapt to your ViT/Mix structure
        def make_hook():
          def get_attn(module, input, output):
            attn_weights.append(module.last_attention) # THIS READS THE STORED ATTENTION MAP FROM THE MULTIHEADATTENTION CLASS
          return get_attn

        h = blk.attn.register_forward_hook(make_hook())
        hooks.append(h)

    # forward pass - trigger teh attention layers so we can use them for visualization
    _ = model(x, x)  # ViTMix: imgs, imgs2

    # remove hooks
    for h in hooks:
        h.remove()

    all_attns = []
    for attn in attn_weights:
        if head_fusion == 'mean':
            attn_fused = attn.mean(dim=1)
        elif head_fusion == 'max':
            attn_fused = attn.max(dim=1)[0]
        else:
            attn_fused = attn.min(dim=1)[0]
        all_attns.append(attn_fused[0])  # remove batch dim

    # rollout computation with residual
    result = torch.eye(all_attns[0].size(0), device = all_attns[0].device)
    for attn in all_attns:

        attn = attn + torch.eye(attn.size(0), device = attn.device)
        attn = attn / attn.sum(dim=-1, keepdim=True)
        result = attn @ result

    rollout = result[0, 1:]  # skip CLS token
    rollout = rollout.reshape(int(np.sqrt(rollout.size(0))), int(np.sqrt(rollout.size(0)))).cpu().numpy()
    return rollout



## ACTUALLY DISPLAYING THE ATTENTION MAPS WITH MATPLOTLIB
def visualize_attention(loader, model, class_names, device, save_dir):
  os.makedirs(save_dir, exist_ok=True)
  model.eval()

  with torch.no_grad():
    for imgs, labels, paths in loader: # the labels here are the GROUND TRUTH FROM THE DATALOADER
      imgs = imgs.to(device)
      labels = labels.to(device)

      # get the models predictions
      logits = model(imgs, imgs)
      predictions = logits.argmax(dim=1)

      for i in range(imgs.size(0)):
        img_tensor = imgs[i].unsqueeze(0)  # batch dim
        rollout_map = attention_rollout(model, img_tensor)

        # original image
        img_pil = Image.open(paths[i]) #.convert("RGB")

        # overlay and show
        overlay = show_rollout_on_image(img_pil, rollout_map)  # returns PIL.Image

        # convert IDs to class names
        gt_label_idx = labels[i].item()
        pred_label_idx = predictions[i].item()

        # label strings
        gt_label = class_names[gt_label_idx]
        pred_label = class_names[pred_label_idx]



        # display
        plt.figure(figsize=(5,5))
        plt.imshow(overlay)
        plt.title(f"Original\nGT {gt_label}\nPredicted class: {pred_label}")
        plt.axis("off")
        plt.show()

        # save
        out_name = os.path.basename(paths[i])
        out_path = os.path.join(save_dir, f"{labels[i].item()}_{out_name}")
        overlay.save(out_path)
        print("Saved", out_path)


############################################################



### PATHWAYS TO DATA ###

# contains original images (no preprocessing)
original_data_dir = "/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification"

# preprocessed images are saved here (resizing and padding)
processed_data_dir = "/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification Resized"

# the final image size that we want
target_img_size = 224


# ONLY RUN THIS FUNCTION ONCE!!!!!
# function calls the preprocess_img_resize_pad function for preprocessing and saves new imgs in a diff location
def preprocess_dataset():

  print("PREPROCESSING STARTED")
  print("")

  # looks for fracture type directories
  for fracture_type in os.listdir(original_data_dir):
    # gets the path for one fracture type
    fracture_path = os.path.join(original_data_dir , fracture_type)
    if not os.path.isdir(fracture_path):
      continue

    # checking that the train and test folders exist - nothing else has happened yet
    for split in ["Train", "Test"]:
      # path to the train or test folder
      split_path = os.path.join(fracture_path, split)
      if not os.path.exists(split_path):
        continue

      # saving in the new destination here
      save_dir = os.path.join(processed_data_dir, fracture_type, split)
      os.makedirs(save_dir, exist_ok = True)

      # looping thru each img file in either the train and test folder per fracture type
      for filename in os.listdir(split_path):

        # load the image
        path = os.path.join(split_path, filename)
        img = Image.open(path) #.convert("RGB")

        # apply the resizing and padding
        processed = preprocess_img_resize_pad(img, target_img_size)

        # save this processed image in the new directory
        save_path = os.path.join(save_dir, filename)
        processed.save(save_path)

        print("PREPROCESSING COMPLETE AND IMGS SAVED IN NEW LOCATION")





# ONLY RUN THIS FUNCTION ONCE!!!!!
# take 20% of the training data and put it in the val folder
def make_val_split(val_ratio=0.2):
  for fracture_type in os.listdir(processed_data_dir):
    # creating paths to the folders
    class_dir = os.path.join(processed_data_dir, fracture_type)
    train_dir = os.path.join(class_dir, "Train")
    val_dir = os.path.join(class_dir, "Val")

    os.makedirs(val_dir, exist_ok=True)

    # CHECK IF WEVE ALREADY MADE THE VAL FOLDER
    if len(os.listdir(val_dir)) > 0:
      print(f"Skipping {fracture_type}: val already created.")
      continue


    images = os.listdir(train_dir)
    random.shuffle(images) # random selection of the images


    val_count = int(len(images)*val_ratio) #val_ratio=0.2; this is where the 20% split happens
    val_images = images[:val_count]

    # moving the images to the val folder
    for img in val_images:
      src = os.path.join(train_dir, img)
      dst = os.path.join(val_dir, img)
      shutil.move(src, dst)



    print(f"{fracture_type}: moved {val_count} images to val/")


############## HELPER FUNCTIONS FOR THE DATALOADER, REFORMATTING THE IMAGE PATHS SO IT CAN BE USED PROPERLY - SAME CODE FROM GRAD-CAM, SCORE-CAM #####

### obtain all images within the 'train' or 'val' subfolder of the  fracture type folder ###
def build_image_list(root, fracture_classes, split): #root= dataset path, split = 'train' or 'val'
  items = [] # holds (image_path, label_index) tupels
  for idx, cls in enumerate(fracture_classes): #idx is integer label assigned to the class in fracture_classes list; cls is class name string ("Avulsion fracture")
    folder = os.path.join(root, cls, split) # creates custom path bc root = "/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification", cls = "Avulsion fracture", split = 'train' or 'val'
    if not os.path.isdir(folder):
      raise FileNotFoundError(f"Expected {folder} to exist")
    # if we do have a folder path, continue on looping thru images
    for ext in('*.png', '*.jpg', '*.jpeg'):
      for p in glob.glob(os.path.join(folder, ext)): # finds all matching file paths (p) in this folder
        items.append((p, idx)) # append file path p and numerical class integer label idx to the list items

  # return looks like this: ("/content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification/Avulsion fracture/train/image_123.jpg", 0) -> 0 is the numerical integer idx for 'Avulsion fracture' folder
  return items # has the path and numerical label for the requested split - 'train' or 'val'

###   converts to RGB, APPLYS TRANSFORMS AND RETURNS A TUPLE ###
class ImageListDataset(Dataset):
  def __init__(self, items, transform=None):
    self.items = items # items is list return by build_image_list
    self.transform = transform
  def __len__(self):
    return len(self.items) # return # of images we have
  def __getitem__(self, idx):
    path, label = self.items[idx] # retirieve the tuple (path, label) from items list, label is numerical integer label
                                  # path is specific: /content/drive/MyDrive/FinalProject_CSC2503/Bone Break Classification/Avulsion fracture/train/img_045.jpg
    img = Image.open(path)#.convert('RGB') # convert to RGB bc most pretrained CNNs expect 3 channel input otherwise we'd get mismatch error
                                          # if we were training from scratch then we dont need to convert to RGB
    # apply transforms here
    if self.transform:
      img = self.transform(img)

    # img is torch.Tensor (3x224x224), label is int (0-9), path is str to the img loc ie./.../Avulsion fracture/train/img_001.jpg for saving gradCAM overlay
    return img, label, path # path so we can save GRAD-CAM overlays next to source images

#######################################################################################################################


# RUN PREPROCESS DATASET ONCE BEFORE TRAINING
# COMMENT THIS LINE OUT ONCE YOUVE DONE IT ONCE
#preprocess_dataset()

# run only once
#make_val_split(val_ratio=0.2)


# can define dataloaders up here
transform = T.Compose([T.Resize(size=(224, 224)), T.Grayscale(num_output_channels=3), T.ToTensor(), T.Normalize(mean=[0.5]*3, std = [0.5]*3)])


# 10 classes of fracture here
fracture_classes = ['Avulsion fracture',
               'Comminuted fracture',
               'Fracture Dislocation',
               'Greenstick fracture',
               'Hairline Fracture',
               'Impacted fracture',
               'Longitudinal fracture',
               'Oblique fracture',
               'Pathological fracture',
               'Spiral Fracture']

# Load the entire fracture dataset (images + labels)
#fracture_dataset = torchvision.datasets.ImageFolder(root=processed_data_dir, transform=transform)
#print("Class mapping:", fracture_dataset.class_to_idx)


train_items = build_image_list(processed_data_dir, fracture_classes, split='Train')
val_items = build_image_list(processed_data_dir, fracture_classes, split='Val')
test_items = build_image_list(processed_data_dir, fracture_classes, split='Test')

train_dataset = ImageListDataset(train_items, transform=transform)
val_dataset = ImageListDataset(val_items, transform=transform)
test_dataset = ImageListDataset(test_items, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers = 2, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers = 2, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers = 2, pin_memory = True)


# print # images for train and val sets
print("Train samples:", len(train_dataset), "Val samples:", len(val_dataset), "Test samples:", len(test_dataset))



# TRAINING FUNCTION
def train_epoch(model, loader, optimizer, device):
  model.train()
  total_loss = 0
  correct = 0

  for imgs, labels, _ in loader:
    imgs, labels = imgs.to(device), labels.to(device)


    # randomize the incoming batch, save it to imgs2 and then pass into model
    # randomizing will make the imgs and img2 batch order diiferent so respective images can be paired together for vitmix
    imgs2 = imgs[torch.randperm(imgs.size(0))]


    logits = model(imgs, imgs2)
    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()*imgs.size(0)
    correct += (logits.argmax(dim=1) == labels).sum().item()

  avg_loss = total_loss/len(loader.dataset)
  accuracy = correct/len(loader.dataset)

  return avg_loss, accuracy




# EVALUATION FUNCTION
def evaluate(model, loader, device):

  model.eval()
  total_loss = 0
  correct = 0

  with torch.no_grad():
    for imgs, labels, _ in loader:
      imgs, labels = imgs.to(device), labels.to(device)

      # DO NOT MIX IN EVALUATE
      # randomize the incoming batch, save it to imgs2 and then pass into model
      # randomizing will make the imgs and img2 batch order diiferent so respective images can be paired together for vitmix
      #imgs2 = imgs[torch.randperm(imgs.size(0))]

      # raw, unormalized outputs of neural net (BEFORE softmax or sigmoid is applied)
      logits = model(imgs, imgs)

      # cross entropy expects logits bc it has a softmax inside it
      loss = F.cross_entropy(logits, labels)

      total_loss += loss.item()*imgs.size(0)
      # the logits.argmax(dim=1) == labels returns smth like tensor([True, True, True])
      correct += (logits.argmax(dim=1) == labels).sum().item()

  avg_loss = total_loss/len(loader.dataset)
  accuracy = correct/len(loader.dataset)

  return avg_loss, accuracy



# TEST FUNCTION
def test(model, test_loader, device):
  model.eval()
  total_loss = 0.0
  correct = 0

  with torch.no_grad():
    for imgs, labels, _ in test_loader:
      imgs, labels = imgs.to(device), labels.to(device)

      # DO NOT MIX IN TEST
      # randomize the incoming batch, save it to imgs2 and then pass into model
      # randomizing will make the imgs and img2 batch order diiferent so respective images can be paired together for vitmix
      #imgs2 = imgs[torch.randperm(imgs.size(0))]

      logits = model(imgs, imgs)

      loss = F.cross_entropy(logits, labels)
      total_loss += loss.item()*imgs.size(0)

      # predictions
      preds = logits.argmax(dim=1)
      correct += (preds == labels).sum().item()

  avg_loss = total_loss/len(test_loader.dataset)
  accuracy = correct/len(test_loader.dataset)

  print(f"Test Loss: {avg_loss:.4f} | Test Accuracy: {accuracy:.4f}")





# RUN THE TRAINING LOOP
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ViT(mix_ratio=0.1).to(device)  # or ViTMiX version
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
best_val_acc = 0



####### CHECK IF WEIGHTS WERE SAVED
start_epoch = 0
best_val_acc = 0.0
epochs = 8  # change this if ur restarting


# Optionally load weights if continuing training or evaluating
if os.path.exists(weights_path):

  #state = torch.load(weights_path, map_location=device)

  checkpoint = torch.load(weights_path, map_location=device)
  model.load_state_dict(checkpoint['model_state_dict'])
  optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
  start_epoch = checkpoint['epoch'] + 1 # resume from next epoch
  best_val_acc = checkpoint['val_acc']
  print(f"✅ Loaded checkpoint from epoch {start_epoch}")
  print("Models loaded successfully!")

else:
  print("NO previously saved models found!")




# BEGIN LOOPING THRU EPOCHS
for epoch in range(start_epoch, epochs):
  train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
  val_loss, val_acc = evaluate(model, val_loader, device)

  print(f"[Epoch {epoch+1}] "
          f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
          f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

  # Save best model
  if val_acc > best_val_acc:
    best_val_acc = val_acc


    # saving epochs, optimizer, state
    torch.save({
      'epoch': epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'val_acc': val_acc
      }, weights_path)

    print(f"✅ Saved best model (epoch {epoch+1}) with val_acc={val_acc:.4f} to {weights_path}")


print("MUST LOAD best saved model before testing: ")

checkpoint = torch.load(weights_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
#linear_head.load_state_dict(checkpoint["linear_state_dict"])

print(f"Loaded best model from epoch {checkpoint['epoch']+1} with val_acc={checkpoint['val_acc']:.4f}")


# RUN TEST FUNCTION ONCE TRAINING AND EVALUATION ARE DONE
test(model, test_loader, device)


# visualize the test data here
# use this variable (defined above): visualization_dir_plots
visualize_attention(test_loader, model, fracture_classes, device, visualization_dir_plots)  # inspect test images

